# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Refresh / Content Opportunity Scoring

The goal is to test whether a simple supervised model can improve content-opportunity prioritization beyond the transparent Week-4 baseline.

The model uses observable March 2026 performance signals and predicts a measurable deterioration in the following month.

The model is treated as decision-support, not as proof that a page should be refreshed.

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

from huggingface_hub import hf_hub_download
from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

print("Imports ready.")

Imports ready.


##Load March
### Prepare the March decision window

March 2026 is the decision window used by the Week-4 baseline.

The dataset is daily × client × content, so the raw March table contains multiple rows for the same content item. We will aggregate it to one row per client-content pair before modeling.

In [4]:
HF_TOKEN = userdata.get("HF_TOKEN")

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_file)

print("March rows:", len(march_df))
print("Date range:", march_df["report_date"].min(), "to", march_df["report_date"].max())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March rows: 9841378
Date range: 2026-03-01 to 2026-03-31


## Load April

This is important because we need a real future outcome, rather than inventing a label.

### Future outcome window

April 2026 is kept separate from the March feature window.

March information is used to make the prediction.

April information is used only to determine whether the predicted opportunity actually appeared.

This separation prevents future information from entering the March features.

In [5]:
april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_df = pd.read_parquet(april_file)

print("April rows:", len(april_df))
print("Date range:", april_df["report_date"].min(), "to", april_df["report_date"].max())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

April rows: 10424730
Date range: 2026-04-01 to 2026-04-30


##Create March features
### 1. Method choice and why

I chose Logistic Regression as the first modeling method.

It fits the Refresh / Content Opportunity Scoring lane because the output is a probability that can be used to rank content opportunities.

It is also interpretable and provides a simple model that can be compared fairly with the Week-4 transparent baseline.

The model uses only March observable signals.

In [6]:
def aggregate_window(data):
    out = (
        data.groupby(
            ["client_hash_id", "content_hash_id"],
            as_index=False
        )
        .agg(
            impressions=("gsc_impressions", "sum"),
            clicks=("gsc_clicks", "sum"),
            sessions=("ga4_sessions", "sum"),
            engaged_sessions=("ga4_engaged_sessions", "sum"),
            avg_position=("gsc_avg_position", "mean")
        )
    )

    out["ctr"] = np.where(
        out["impressions"] > 0,
        out["clicks"] / out["impressions"],
        np.nan
    )

    out["engagement_rate"] = np.where(
        out["sessions"] > 0,
        out["engaged_sessions"] / out["sessions"],
        np.nan
    )

    return out


march_features = aggregate_window(march_df)

print("March modeling rows:", len(march_features))
march_features.head()

March modeling rows: 331437


,client_hash_id,content_hash_id,impressions,clicks,sessions,engaged_sessions,avg_position,ctr,engagement_rate
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,0.0,0.0,NaN,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,0.0,0.0,NaN,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,0.0,0.0,NaN,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,0.0,0.0,9.0,0.0,NaN
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,0.0,0.0,NaN,NaN,NaN


##Create April outcome
### Future opportunity label

The future label is:

A content item is marked as a future opportunity when:

1. it had at least 500 March impressions;
2. March CTR was measurable;
3. April CTR was measurable; and
4. April CTR was at least 20% lower than March CTR.

This creates a real future outcome from the April window rather than using the Week-4 score as the truth.

The 20% threshold is a chosen policy threshold for this baseline experiment, not a universal definition of content failure.

In [7]:
april_features = aggregate_window(april_df)

model_df = march_features.merge(
    april_features,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    suffixes=("_march", "_april")
)

model_df["future_opportunity"] = (
    (model_df["impressions_march"] >= 500) &
    (model_df["ctr_march"].notna()) &
    (model_df["ctr_april"].notna()) &
    (model_df["ctr_april"] <= model_df["ctr_march"] * 0.80)
).astype(int)

print("Model rows:", len(model_df))
print("Future opportunities:", model_df["future_opportunity"].sum())
print("Opportunity rate:", model_df["future_opportunity"].mean())

Model rows: 331436
Future opportunities: 35181
Opportunity rate: 0.10614718980436645


##Check the target

In [8]:
print(
    model_df["future_opportunity"]
    .value_counts()
    .rename({0: "No opportunity", 1: "Opportunity"})
)

print("\nTarget percentage:")
print(
    (model_df["future_opportunity"].value_counts(normalize=True) * 100)
    .round(2)
)

future_opportunity
No opportunity    296255
Opportunity        35181
Name: count, dtype: int64

Target percentage:
future_opportunity
0    89.39
1    10.61
Name: proportion, dtype: float64


##Build the model features
### Features

The model uses March-only observable signals:

- impressions
- clicks
- CTR
- sessions
- engagement rate
- average position

The April columns are used only to create the future label.

The Week-4 reason code, action, and baseline score are not model features because they are outputs of the previous rule and could create circular learning.

In [9]:
feature_columns = [
    "impressions_march",
    "clicks_march",
    "ctr_march",
    "sessions_march",
    "engagement_rate_march",
    "avg_position_march"
]

X = model_df[feature_columns].copy()
y = model_df["future_opportunity"].copy()
groups = model_df["client_hash_id"].copy()

print("Features:")
print(feature_columns)

print("\nRows:", len(X))

Features:
['impressions_march', 'clicks_march', 'ctr_march', 'sessions_march', 'engagement_rate_march', 'avg_position_march']

Rows: 331436


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The future label comes from April, while the features come from March.

For the train/test split, clients are kept together.

This means observations from the same client are not intentionally placed in both training and testing groups.

A grouped split is appropriate because many content items can belong to the same client, and allowing the same client to appear in both groups could make the test set easier than a genuinely unseen-client evaluation.

In [10]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

test_model_df = model_df.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Training rows: 300879
Test rows: 30557
Training clients: 44
Test clients: 11


### Verify no client overlap

In [11]:
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

overlap = train_clients.intersection(test_clients)

print("Client overlap:", len(overlap))

if len(overlap) == 0:
    print("Grouped split check passed.")
else:
    print("WARNING: client overlap detected.")

Client overlap: 0
Grouped split check passed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The model is a Logistic Regression classifier.

A preprocessing pipeline is used so that missing values are imputed using training data and numeric features are standardized before fitting the model.

The model produces a probability score for future opportunity.

In [12]:
model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

model_probability = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

Model trained successfully.


### Model evaluation/matrics

The main evaluation metric is Average Precision because this is a ranking problem with a potentially imbalanced future-opportunity class.

ROC AUC is also reported as a secondary ranking metric.

Precision at 20 is included because the practical use case is a review queue where only a limited number of pages may be reviewed.

In [13]:
def precision_at_k(y_true, scores, k=20):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["y"].mean()


model_ap = average_precision_score(
    y_test,
    model_probability
)

model_auc = roc_auc_score(
    y_test,
    model_probability
)

model_p20 = precision_at_k(
    y_test,
    model_probability,
    k=20
)

print("Model Average Precision:", round(model_ap, 4))
print("Model ROC AUC:", round(model_auc, 4))
print("Model Precision@20:", round(model_p20, 4))

Model Average Precision: 0.4526
Model ROC AUC: 0.9265
Model Precision@20: 0.4


## Recreate Week-4 baseline

This is the important comparison.
We are not using the baseline score as a model feature.
We're using it only as the competing ranking system.

The Week-4 baseline was a transparent rule-based prioritization system.

For a fair comparison, the baseline is rebuilt from the same March observable signals and evaluated only on the same held-out test observations used for the Logistic Regression model.

The baseline is therefore treated as a competing ranking method, not as the target.

In [14]:
baseline_df = model_df.copy()

baseline_df["baseline_action_score"] = 0.0

# Low CTR with meaningful visibility
baseline_df.loc[
    (
        (baseline_df["impressions_march"] >= 500) &
        (baseline_df["ctr_march"] < 0.005)
    ),
    "baseline_action_score"
] += 40

# Low engagement with meaningful sessions
baseline_df.loc[
    (
        (baseline_df["sessions_march"] >= 30) &
        (baseline_df["engagement_rate_march"].notna()) &
        (baseline_df["engagement_rate_march"] < 0.30)
    ),
    "baseline_action_score"
] += 30

# Demand exists
baseline_df.loc[
    (
        (baseline_df["impressions_march"] >= 100) &
        (baseline_df["clicks_march"] > 0)
    ),
    "baseline_action_score"
] += 20

# Strong visibility
baseline_df.loc[
    baseline_df["impressions_march"] >= 500,
    "baseline_action_score"
] += 10

baseline_score_test = (
    baseline_df.iloc[test_idx]["baseline_action_score"]
    .to_numpy()
)

print("Baseline scores ready.")

Baseline scores ready.


##Baseline metrics

In [15]:
baseline_ap = average_precision_score(
    y_test,
    baseline_score_test
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_score_test
)

baseline_p20 = precision_at_k(
    y_test,
    baseline_score_test,
    k=20
)

print("Baseline Average Precision:", round(baseline_ap, 4))
print("Baseline ROC AUC:", round(baseline_auc, 4))
print("Baseline Precision@20:", round(baseline_p20, 4))

Baseline Average Precision: 0.4981
Baseline ROC AUC: 0.9356
Baseline Precision@20: 0.45


##Model vs baseline table
Both methods are evaluated on:

- the same held-out observations;
- the same future opportunity label;
- the same ranking metrics.

This makes the comparison directly interpretable.

In [16]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "average_precision": [
        baseline_ap,
        model_ap
    ],
    "roc_auc": [
        baseline_auc,
        model_auc
    ],
    "precision_at_20": [
        baseline_p20,
        model_p20
    ]
})

comparison.round(4)

,method,average_precision,roc_auc,precision_at_20
0,Week-4 baseline,0.4981,0.9356,0.45
1,Logistic Regression,0.4526,0.9265,0.40


## Winner

In [17]:
if model_ap > baseline_ap:
    result = "Logistic Regression performed better on Average Precision."
elif model_ap < baseline_ap:
    result = "The Week-4 baseline performed better on Average Precision."
else:
    result = "The model and baseline had the same Average Precision."

print(result)

The Week-4 baseline performed better on Average Precision.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The error analysis focuses on observations where the model and future outcome disagree.

These errors matter because a ranking system can appear useful overall while still producing weak recommendations for individual observations.

The analysis below separates:

- false positives: predicted opportunity but no future opportunity;
- false negatives: future opportunity but low predicted probability.

In [18]:
error_df = test_model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_march",
        "clicks_march",
        "ctr_march",
        "sessions_march",
        "engagement_rate_march",
        "avg_position_march",
        "future_opportunity"
    ]
].copy()

error_df["model_probability"] = model_probability

error_df["predicted_opportunity"] = (
    error_df["model_probability"] >= 0.50
).astype(int)

error_df["error_type"] = "correct"

error_df.loc[
    (
        (error_df["predicted_opportunity"] == 1) &
        (error_df["future_opportunity"] == 0)
    ),
    "error_type"
] = "false_positive"

error_df.loc[
    (
        (error_df["predicted_opportunity"] == 0) &
        (error_df["future_opportunity"] == 1)
    ),
    "error_type"
] = "false_negative"

print(error_df["error_type"].value_counts())

error_type
correct           26695
false_positive     2351
false_negative     1511
Name: count, dtype: int64


### False positives

In [19]:
false_positives = (
    error_df[
        error_df["error_type"] == "false_positive"
    ]
    .sort_values("model_probability", ascending=False)
)

print("False positives:", len(false_positives))

false_positives.head(10)

False positives: 2351


,client_hash_id,content_hash_id,impressions_march,clicks_march,ctr_march,sessions_march,engagement_rate_march,avg_position_march,future_opportunity,model_probability,predicted_opportunity,error_type
305898,client_e547b89c05043229,content_f86f77b3ebdc05ee,105420,548,0.005198,437.0,0.038902,3.942110,0,1.0,1,false_positive
305393,client_e547b89c05043229,content_eadb33b5df496f4a,617124,5668,0.009185,2730.0,0.082051,2.383011,0,1.0,1,false_positive
302712,client_e547b89c05043229,content_9ef3d7516483e665,89229,92,0.001031,42.0,0.190476,2.481596,0,1.0,1,false_positive
303442,client_e547b89c05043229,content_b2b85c287474668d,65304,61,0.000934,41.0,0.024390,1.541702,0,1.0,1,false_positive
303718,client_e547b89c05043229,content_bbf12024947e4fc6,61158,286,0.004676,210.0,0.057143,7.238398,0,1.0,1,false_positive
304206,client_e547b89c05043229,content_c9a0c2fdbdbfb562,65681,739,0.011251,293.0,0.040956,2.446912,0,1.0,1,false_positive
304298,client_e547b89c05043229,content_cc26620b2cbb837f,59109,150,0.002538,124.0,0.032258,5.312944,0,1.0,1,false_positive
299400,client_e547b89c05043229,content_45be1a7ea8833f6a,63730,50,0.000785,34.0,0.117647,3.794223,0,1.0,1,false_positive
299797,client_e547b89c05043229,content_4ffe18112a5642e3,186983,586,0.003134,364.0,0.164835,2.331060,0,1.0,1,false_positive
297035,client_e547b89c05043229,content_044eae5cec1e4ac1,72124,457,0.006336,179.0,0.150838,2.029767,0,1.0,1,false_positive


###False negatives

In [20]:
false_negatives = (
    error_df[
        error_df["error_type"] == "false_negative"
    ]
    .sort_values("model_probability", ascending=False)
)

print("False negatives:", len(false_negatives))

false_negatives.head(10)

False negatives: 1511


,client_hash_id,content_hash_id,impressions_march,clicks_march,ctr_march,sessions_march,engagement_rate_march,avg_position_march,future_opportunity,model_probability,predicted_opportunity,error_type
302234,client_e547b89c05043229,content_9316b547683fd3dc,1065,7,0.006573,9.0,0.0,3.086981,1,0.499911,0,false_negative
298308,client_e547b89c05043229,content_271b0445a16658c1,1221,3,0.002457,2.0,0.0,4.269888,1,0.499901,0,false_negative
298518,client_e547b89c05043229,content_2d1e1ac958dbd5d2,1270,0,0.000000,1.0,0.0,9.006811,1,0.498861,0,false_negative
304201,client_e547b89c05043229,content_c9794d3f38fbafbf,1193,4,0.003353,4.0,0.0,18.201865,1,0.498818,0,false_negative
298129,client_e547b89c05043229,content_223390e4c0848c2d,1198,3,0.002504,3.0,0.0,4.578335,1,0.498687,0,false_negative
304715,client_e547b89c05043229,content_d84ba18474809e08,1322,0,0.000000,0.0,NaN,34.096084,1,0.498503,0,false_negative
286104,client_c182d11e4862a37d,content_21f1e2757b35bcd6,1257,1,0.000796,1.0,0.0,10.425976,1,0.498165,0,false_negative
302146,client_e547b89c05043229,content_904b8f99e1b4846b,1203,2,0.001663,3.0,0.0,4.715487,1,0.497728,0,false_negative
302957,client_e547b89c05043229,content_a540c2357e5fbc4f,1204,3,0.002492,2.0,0.0,4.069976,1,0.497241,0,false_negative
302700,client_e547b89c05043229,content_9ea9aa57735ec347,1292,0,0.000000,1.0,0.0,30.178332,1,0.497204,0,false_negative


## Feature interpretation

Logistic Regression provides coefficients that show the direction of the relationship between each standardized feature and the predicted opportunity probability.

These coefficients describe the model's learned association.

They do not prove that changing a signal will cause future performance to change.

In [21]:
classifier = model.named_steps["classifier"]

coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": classifier.coef_[0]
})

coefficients["absolute_effect"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "absolute_effect",
    ascending=False
)

coefficients

,feature,coefficient,absolute_effect
0,impressions_march,2.474316,2.474316
3,sessions_march,0.260284,0.260284
1,clicks_march,0.115976,0.115976
4,engagement_rate_march,0.097286,0.097286
5,avg_position_march,-0.013414,0.013414
2,ctr_march,0.000884,0.000884


## Leakage check
The model must not use information from the future outcome window.

The following fields are intentionally excluded from the model features:

- April metrics;
- future opportunity label;
- Week-4 action;
- Week-4 reason code;
- Week-4 baseline score;
- product decision flags;
- client names;
- URLs;
- private queries.

The model features are all March observable signals.

In [22]:
forbidden_terms = [
    "april",
    "future",
    "label",
    "target",
    "reason_code",
    "action",
    "baseline_action_score",
    "health_score",
    "priority_score",
    "refresh_tier",
    "flag"
]

feature_names_lower = [
    col.lower()
    for col in feature_columns
]

found_forbidden = [
    term
    for term in forbidden_terms
    if any(term in col for col in feature_names_lower)
]

print("Potential forbidden terms:", found_forbidden)

if not found_forbidden:
    print("Leakage feature-name check passed.")
else:
    print("Review required.")

Potential forbidden terms: []
Leakage feature-name check passed.


### Verify feature timing

In [23]:
print("Feature window: March 2026")
print("Target window: April 2026")
print("Feature columns:")
print(feature_columns)

print("\nApril columns are not present in X:")
print(
    not any(
        "april" in col.lower()
        for col in X.columns
    )
)

Feature window: March 2026
Target window: April 2026
Feature columns:
['impressions_march', 'clicks_march', 'ctr_march', 'sessions_march', 'engagement_rate_march', 'avg_position_march']

April columns are not present in X:
True


## Top recommendations

The model probability is used to create a ranked review queue.

These are recommendations for human review, not automatic refresh decisions.

In [24]:
recommendations = test_model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_march",
        "clicks_march",
        "ctr_march",
        "sessions_march",
        "engagement_rate_march",
        "avg_position_march",
        "future_opportunity"
    ]
].copy()

recommendations["model_probability"] = model_probability

recommendations = recommendations.sort_values(
    "model_probability",
    ascending=False
).reset_index(drop=True)

recommendations["rank"] = recommendations.index + 1

recommendations.head(20)

,client_hash_id,content_hash_id,impressions_march,clicks_march,ctr_march,sessions_march,engagement_rate_march,avg_position_march,future_opportunity,model_probability,rank
0,client_e547b89c05043229,content_545bb6cc7081ded3,122905,287,0.002335,249.0,0.120482,2.615390,0,1.0,1
1,client_e547b89c05043229,content_45be1a7ea8833f6a,63730,50,0.000785,34.0,0.117647,3.794223,0,1.0,2
2,client_e547b89c05043229,content_629b2d2f28c32b39,59198,306,0.005169,271.0,0.099631,4.068347,0,1.0,3
3,client_e547b89c05043229,content_b2b85c287474668d,65304,61,0.000934,41.0,0.024390,1.541702,0,1.0,4
4,client_e547b89c05043229,content_4ffe18112a5642e3,186983,586,0.003134,364.0,0.164835,2.331060,0,1.0,5
5,client_e547b89c05043229,content_963de14b1f58978f,97312,482,0.004953,354.0,0.031073,3.753124,1,1.0,6
6,client_e547b89c05043229,content_18f0847d6628f8c6,48911,703,0.014373,684.0,0.152047,3.825143,1,1.0,7
7,client_e547b89c05043229,content_f86f77b3ebdc05ee,105420,548,0.005198,437.0,0.038902,3.942110,0,1.0,8
8,client_e547b89c05043229,content_044eae5cec1e4ac1,72124,457,0.006336,179.0,0.150838,2.029767,0,1.0,9
9,client_e547b89c05043229,content_ec2e0346994fb5a5,245276,1480,0.006034,816.0,0.110294,2.854514,1,1.0,10


### What could make recommendations wrong?


In [25]:
def what_would_make_it_wrong(row):
    if row["impressions_march"] < 500:
        return "Limited search exposure makes the measured CTR movement less stable."

    if pd.isna(row["ctr_march"]):
        return "March CTR was unavailable, so the opportunity evidence is incomplete."

    if pd.isna(row["engagement_rate_march"]):
        return "GA4 engagement coverage is incomplete for this observation."

    return "Seasonality, search-intent changes, measurement coverage, or other unobserved factors may explain the future movement."


top20 = recommendations.head(20).copy()

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

top20[
    [
        "rank",
        "model_probability",
        "future_opportunity",
        "impressions_march",
        "ctr_march",
        "what_would_make_it_wrong"
    ]
]

,rank,model_probability,future_opportunity,impressions_march,ctr_march,what_would_make_it_wrong
0,1,1.0,0,122905,0.002335,"Seasonality, search-intent changes, measuremen..."
1,2,1.0,0,63730,0.000785,"Seasonality, search-intent changes, measuremen..."
2,3,1.0,0,59198,0.005169,"Seasonality, search-intent changes, measuremen..."
3,4,1.0,0,65304,0.000934,"Seasonality, search-intent changes, measuremen..."
4,5,1.0,0,186983,0.003134,"Seasonality, search-intent changes, measuremen..."
5,6,1.0,1,97312,0.004953,"Seasonality, search-intent changes, measuremen..."
6,7,1.0,1,48911,0.014373,"Seasonality, search-intent changes, measuremen..."
7,8,1.0,0,105420,0.005198,"Seasonality, search-intent changes, measuremen..."
8,9,1.0,0,72124,0.006336,"Seasonality, search-intent changes, measuremen..."
9,10,1.0,1,245276,0.006034,"Seasonality, search-intent changes, measuremen..."


## Final findings

The experiment compares a Logistic Regression ranking model with the transparent Week-4 baseline.

Both methods are evaluated on the same held-out client groups and against the same April future-opportunity outcome.

The model comparison should be interpreted using the measured Average Precision, ROC AUC, and Precision@20 values rather than complexity alone.

If Logistic Regression performs better, the result suggests that combining observable March signals can improve future-opportunity ranking beyond the simple Week-4 rule.

If the Week-4 baseline performs better, that is also a useful result: the added model complexity did not improve the measured decision metric.

The error analysis shows where the model makes false-positive and false-negative recommendations.

These results are directional decision-support evidence. They do not prove that refreshing a page causes recovery or that any individual signal causes future performance.

###Automatically generate the conclusion

In [26]:
print("===== FINAL MODEL VS BASELINE =====")
print(comparison.round(4))

print("\n===== INTERPRETATION =====")

if model_ap > baseline_ap:
    print(
        "Observed result: Logistic Regression has higher Average Precision "
        "than the Week-4 baseline on the held-out test set."
    )
elif model_ap < baseline_ap:
    print(
        "Observed result: the Week-4 baseline has higher Average Precision "
        "than Logistic Regression on the held-out test set."
    )
else:
    print(
        "Observed result: both methods have the same Average Precision "
        "on the held-out test set."
    )

print(
    "\nThis is directional decision-support evidence and does not establish "
    "that refreshing content causes recovery."
)

===== FINAL MODEL VS BASELINE =====
                method  average_precision  roc_auc  precision_at_20
0      Week-4 baseline             0.4981   0.9356             0.45
1  Logistic Regression             0.4526   0.9265             0.40

===== INTERPRETATION =====
Observed result: the Week-4 baseline has higher Average Precision than Logistic Regression on the held-out test set.

This is directional decision-support evidence and does not establish that refreshing content causes recovery.


## 5. Self-check

- [x] Method choice is explained.
- [x] Logistic Regression fits the Refresh / Content Opportunity Scoring lane.
- [x] March is used as the feature/decision window.
- [x] April is used only for the future outcome.
- [x] A real future outcome is used instead of a synthetic model target.
- [x] Features do not include the future window.
- [x] Week-4 baseline is treated as a competing ranking method, not as the target.
- [x] Train/test split is grouped by client.
- [x] There is no client overlap between train and test.
- [x] The model is trained only on the training set.
- [x] The baseline and model are evaluated on the same test observations.
- [x] Average Precision is reported.
- [x] ROC AUC is reported.
- [x] Precision@20 is reported.
- [x] Model-vs-baseline table is shown.
- [x] False positives are inspected.
- [x] False negatives are inspected.
- [x] Feature coefficients are interpreted.
- [x] Leakage checks are performed.
- [x] Top recommendations are shown.
- [x] "What would make it wrong" is documented.
- [x] No client names, URLs, or private queries are used.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] The notebook should be run top-to-bottom before committing.